In [ ]:
from pathlib import Path
import requests
import yaml

import numpy as np
import pandas as pd

from benchmark_data_leakage.utils import find_repo_root
from benchmark_data_leakage.chembl_requester import ChEMBLRequester

repo_root = find_repo_root()
data_dir = repo_root / "data"
processed_dir = data_dir / "processed"
processed_dir.mkdir(exist_ok=True, parents=True)
out_dir = data_dir / "out"
out_dir.mkdir(exist_ok=True, parents=True)

with open(repo_root / "chembl_db.yaml") as f:
    chembl_db = yaml.safe_load(f)
chembl_req = ChEMBLRequester(**chembl_db)

### Create list of test targets and prepare for clustering

In [ ]:
TARGET_MAPPING_FEP_4 = {
    # Keys are folder names as in https://github.com/openforcefield/protein-ligand-benchmark/tree/0.2.1/data
    # Values are uniprot_id
    "2019-12-09_p38": "Q16539",
    "2019-09-23_jnk1": "P45983", 
    "2020-02-07_tyk2": "P29597", 
    "2019-12-13_cdk2": "P24941",
}

TARGET_MAPPING_OPEN_FE = {
    # Keys are `system group` and `system name` as in this file https://github.com/OpenFreeEnergy/IndustryBenchmarks2024/blob/ff41e5ad0fda89b352341e3f6511bee25db0000a/industry_benchmarks/analysis/processed_results/combined_pymbar3_calculated_dg_data.csv
    # Values are uniprot_id
    # Mapped performed by looking at https://github.com/OpenFreeEnergy/IndustryBenchmarks2024/tree/ff41e5ad0fda89b352341e3f6511bee25db0000a/industry_benchmarks/input_structures/original_structures/*/subset_metadata.csv
    # and mapping `Reference PDB` column to uniprot_id
    ('charge_annihilation_set', 'cdk2'): 'P24941',
    ('charge_annihilation_set', 'dlk'): 'Q12852',
    ('charge_annihilation_set', 'egfr'): 'P00533',
    ('charge_annihilation_set', 'ephx2'): 'P34913',
    ('charge_annihilation_set', 'irak4_s2'): 'Q9NWZ3',
    ('charge_annihilation_set', 'irak4_s3'): 'Q9NWZ3',
    ('charge_annihilation_set', 'itk'): 'Q08881',
    ('charge_annihilation_set', 'jak1'): 'P23458',
    ('charge_annihilation_set', 'jnk1'): 'P45983',
    ('charge_annihilation_set', 'ptp1b'): 'P18031',
    ('charge_annihilation_set', 'thrombin'): 'P00734',
    ('charge_annihilation_set', 'tyk2'): 'P29597',
    ('fragments', 'hsp90_2rings'): 'P07900',
    ('fragments', 'hsp90_single_ring'): 'P07900',
    ('fragments', 'jak2_set1'): 'O60674',
    ('fragments', 'jak2_set2'): 'O60674',
    ('fragments', 'liga'): 'Q9AIU7',
    ('fragments', 'mcl1'): 'Q07820',
    ('fragments', 'mup1'): 'P02762',
    ('fragments', 'p38'): 'Q16539',
    ('fragments', 't4_lysozyme'): 'P00720',
    ('jacs_set', 'bace'): 'P56817',
    ('jacs_set', 'cdk2'): 'P24941',
    ('jacs_set', 'jnk1'): 'P45983',
    ('jacs_set', 'mcl1'): 'Q07820',
    ('jacs_set', 'p38'): 'Q16539',
    ('jacs_set', 'ptp1b'): 'P18031',
    ('jacs_set', 'thrombin'): 'P00734',
    ('jacs_set', 'tyk2'): 'P29597',
    ('janssen_bace', 'bace_ciordia_prospective'): 'P56817',
    ('janssen_bace', 'bace_p3_arg368_in'): 'P56817',
    ('janssen_bace', 'ciordia_retro'): 'P56817',
    ('janssen_bace', 'keranen_p2'): 'P56817',
    ('mcs_docking_set', 'hne'): 'P08246',
    ('mcs_docking_set', 'renin'): 'P00797',
    ('merck', 'cdk8'): 'P49336',
    ('merck', 'cmet'): 'P08581',
    ('merck', 'eg5'): 'P52732',
    ('merck', 'hif2a'): 'Q99814',
    ('merck', 'pfkfb3'): 'Q16875',
    ('merck', 'shp2'): 'Q06124',
    ('merck', 'syk'): 'P43405',
    ('merck', 'tnks2'): 'Q9H2K2',
    ('miscellaneous_set', 'btk'): 'Q06187',
    ('miscellaneous_set', 'cdk8'): 'P49336',
    ('miscellaneous_set', 'faah'): 'P97612',
    ('miscellaneous_set', 'galectin'): 'P17931',
    ('miscellaneous_set', 'hiv1_protease'): 'Q5RZ09',
    ('scaffold_hopping_set', 'bace1'): 'P56817',
    ('scaffold_hopping_set', 'factor_xa'): 'P00742',
    ('water_set', 'brd4'): 'O60885',
    ('water_set', 'chk1'): 'O14757',
    ('water_set', 'hsp90_kung'): 'P07900',
    ('water_set', 'hsp90_woodhead'): 'P07900',
    ('water_set', 'scyt_dehyd'): 'P56221',
    ('water_set', 'taf12'): 'P21675',  # Note this is TAF1(2) and not TAF12 as per the subset_metadata.csv
    ('water_set', 'thrombin'): 'P00734',
    ('water_set', 'urokinase'): 'P00749',
}

TEST_SET_UNIPROT_IDS = set(TARGET_MAPPING_FEP_4.values()) | set(TARGET_MAPPING_OPEN_FE.values())

In [ ]:
def get_fasta_sequence_from_uniprot(uniprot_id: str) -> str:
    url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
    response = requests.get(url)
    response.raise_for_status()
    
    lines = response.text.strip().splitlines()
    sequence = "".join(line.strip() for line in lines if not line.startswith(">"))
    
    return sequence

# Get uniprot_id -> fasta sequence from chembl
df_protein_data = chembl_req.get_uniprot_id_component_data()
df_protein_data = pd.DataFrame(df_protein_data)

# Add missing fasta sequences not in chembl
uniprot_ids_not_in_chembl = [ui for ui in TEST_SET_UNIPROT_IDS if ui not in df_protein_data["uniprot_id"].values]
print(f"Missing: {uniprot_ids_not_in_chembl}")
extra_data = []
for uniprot_id in uniprot_ids_not_in_chembl:
    fasta_seq = get_fasta_sequence_from_uniprot(uniprot_id=uniprot_id)
    extra_data.append({
        "uniprot_id": uniprot_id,
        "sequence": fasta_seq
    })
extra_data = pd.DataFrame(extra_data)
df_protein_data = pd.concat([df_protein_data, extra_data], ignore_index=True)

In [ ]:
# Save as one fasta file
with open(processed_dir / "fasta_seqs_for_clustering.fasta", "w") as f:
    for _, row in df_protein_data.iterrows():
        f.write(f">{row['uniprot_id']}\n{row['sequence']}\n")

### Run clustering

In the terminal go into data/processed and run
```
mmseqs easy-cluster fasta_seqs_for_clustering.fasta cluster_res cluster_tmp --min-seq-id 0.9 --cov-mode 0 -c 0.01
```

### Use clustering to extract split

In [ ]:
def split_cluster_by_test_set(
    tsv_path: str,
    test_set_uniprot_ids: set,
    val_fraction: float = 0.1,
    random_seed: int = 42,
) -> tuple[list, list, list]:
    clusters = pd.read_csv(tsv_path, sep='\t', header=None, names=['cluster_name', 'uniprot_id'])

    # Find all cluster names that contain at least one test set ID
    test_clusters = set(
        clusters.loc[clusters['uniprot_id'].isin(test_set_uniprot_ids), 'cluster_name']
    )

    in_test = []
    remaining_rows = []

    for _, row in clusters.iterrows():
        entry = (row['cluster_name'], row['uniprot_id'])
        if row['cluster_name'] in test_clusters:
            in_test.append(entry)
        else:
            remaining_rows.append(entry)

    # Assign remaining clusters to val or train at cluster level
    remaining_cluster_names = sorted(set(cluster for cluster, _ in remaining_rows))
    rng = np.random.default_rng(random_seed)
    rng.shuffle(remaining_cluster_names)
    n_val = round(len(remaining_cluster_names) * val_fraction)
    val_clusters = set(remaining_cluster_names[:n_val])

    in_val = [(c, u) for c, u in remaining_rows if c in val_clusters]
    in_train = [(c, u) for c, u in remaining_rows if c not in val_clusters]

    return in_test, in_val, in_train


in_test_cluster, in_val_cluster, in_train_cluster = split_cluster_by_test_set(
    tsv_path=processed_dir / "cluster_res_cluster.tsv",
    test_set_uniprot_ids=TEST_SET_UNIPROT_IDS,
    val_fraction=0.1,
    random_seed=42,
)

_, uniprot_id_test = zip(*in_test_cluster)
_, uniprot_id_val = zip(*in_val_cluster)
_, uniprot_id_train = zip(*in_train_cluster)

In [ ]:
split_lookup = {}
for ui in uniprot_id_test:
    split_lookup[ui] = "test"
for ui in uniprot_id_val:
    split_lookup[ui] = "val"
for ui in uniprot_id_train:
    split_lookup[ui] = "train"

In [ ]:
out_data = {}
for ui in uniprot_id_test:
    out_data[ui] = "test"
for ui in uniprot_id_val:
    out_data[ui] = "val"
for ui in uniprot_id_train:
    out_data[ui] = "train"
out_data = pd.Series(data=out_data.values(), name="data_split", index=out_data.keys())
out_data.index.name = "uniprot_id"
out_data.to_csv(out_dir / "FEP_data_split.csv")